# 17-01 生成步长与想象中的策略学习

本实验对应第 17 章，沿着“构造带噪未来表示 → 两个小步形成大步训练目标 → 策略通过世界模型产生想象轨迹 → 回报改变策略概率”运行。依赖 NumPy、Matplotlib 和 PyTorch，CPU 即可。

所有状态、动作和奖励都是可直接检查的教学数字。实验不训练视频 Transformer，也不是 Dreamer 4 完整复现；它的目的，是让每个输入和输出在三阶段训练逻辑中都有明确位置。


## 1. 先区分环境时间与生成进度

固定一帧干净未来表示和一份高斯噪声。生成进度 $\tau$ 从 0 走到 1 时，中间表示从噪声端沿直线靠近干净表示。整条曲线都属于**同一个未来帧**，并不是机器人连续经历了六个环境时刻。

下面不训练网络，只把训练时构造带噪样本的公式逐点画出来。


In [ ]:
import numpy as np  # 计算带噪表示、想象轨迹和回报
import matplotlib.pyplot as plt  # 绘制生成进度与策略学习结果
import torch  # 用梯度演示策略概率更新
clean_representation = np.array([0.80, -0.20])  # 设置真实未来帧编码出的二维干净表示
noise_representation = np.array([-0.60, 0.50])  # 固定一份用于演示的二维噪声
progress_values = np.linspace(0.0, 1.0, 6)  # 在同一帧内部选取六个生成进度
mixed_representations = np.array([(1.0 - tau) * noise_representation + tau * clean_representation for tau in progress_values])  # 按线性路径构造各进度的带噪表示
print(f'生成进度 0 的表示：{mixed_representations[0]}，它等于噪声端。')  # 核对路径起点
print(f'生成进度 1 的表示：{mixed_representations[-1]}，它等于干净未来表示。')  # 核对路径终点
plt.figure(figsize=(7, 3.5))  # 创建两个表示分量随生成进度变化的图
plt.plot(progress_values, mixed_representations[:, 0], 'o-', label='latent component 1')  # 绘制第一个潜空间分量
plt.plot(progress_values, mixed_representations[:, 1], 's-', label='latent component 2')  # 绘制第二个潜空间分量
plt.xlabel('generation progress tau')  # 标明横轴是单帧内部的生成进度
plt.ylabel('latent value')  # 标明纵轴是潜空间数值而非物理位置
plt.title('One future frame: noise to clean representation')  # 强调整条曲线属于同一未来帧
plt.grid(alpha=0.25)  # 添加浅色网格便于读取中间值
plt.legend()  # 区分两个表示分量
plt.show()  # 输出带噪路径


### 这张图不是机器人运动轨迹

横轴的 0、0.2、0.4……表示同一未来帧的恢复进度。到达 $\tau=1$ 后，才得到这一帧的干净表示；模型随后才能把它加入历史，开始生成下一个环境时刻。

训练时随机抽取某个 $\tau$，把对应带噪表示、历史、动作和生成条件交给网络，干净表示留作答案。图中直线是人为规定的加噪路径，不是网络已经学会的去噪结果。


## 2. 两个小步怎样形成一个大步目标

当前带噪表示的一个分量是 0.2，总生成步长为 0.5。第一次小步在起点查询方向，走到中间点以后必须再次查询方向。一次大步学习两个小步方向的平均值。

这段代码只核对 Shortcut forcing 的目标构造。两条路径终点一致是算术设计，不代表视频生成器已经训练完成。


In [ ]:
import numpy as np  # 计算生成步骤与轨迹回报
import matplotlib.pyplot as plt  # 显示步骤与概率变化
import torch  # 用梯度演示一次简化策略更新
tau = 0.0  # 指定当前生成进度
d = 0.5  # 指定一次大步覆盖的进度区间
x = 0.2  # 指定当前带噪表示的一个分量
first_direction = 0.8  # 假设模型在起点预测的小步方向
midpoint = x + (d / 2) * first_direction  # 按第一小步到达中间点
second_direction = 0.4  # 假设模型在中间点重新查询得到的方向
endpoint = midpoint + (d / 2) * second_direction  # 按第二小步到达区间终点
target_direction = (first_direction + second_direction) / 2  # 构造一次大步应学习的平均方向
big_endpoint = x + d * target_direction  # 检查大步目标对应的终点
fig, ax = plt.subplots(figsize=(7, 3.5))  # 创建生成进度示意图
ax.plot([tau, tau+d/2, tau+d], [x, midpoint, endpoint], "o-", label="Two half steps")  # 显示两次查询之间的中间状态
ax.plot([tau, tau+d], [x, big_endpoint], "s--", label="One-step target", markersize=7)  # 显示大步学习目标而非已训练网络输出
ax.set(xlabel="Generation progress", ylabel="One latent component")  # 防止把潜表示变化误读为机器人速度
ax.legend()  # 区分两小步与大步目标
plt.show()  # 显示它们的终点关系
print(f"中间点={midpoint:.2f}；两步终点={endpoint:.2f}；大步方向目标={target_direction:.2f}")  # 输出每项算术结果
assert np.isclose(endpoint, big_endpoint)  # 核对构造的监督目标与两小步一致


### 输出中的四个数分别代表什么

`中间点=0.40` 来自起点 0.20 加上第一小步；第二个方向是在这个新位置重新得到的。两小步最后到 0.50，平均方向是 0.60，因此一次 0.5 的大步也以 0.50 为训练目标。

图的横轴仍是生成进度，不是物理时间。Shortcut forcing 的用途是让模型有机会用较少网络调用生成一帧；它不会让机器人跳过实际接触，也不能消除跨环境时间连续生成的全部误差。


## 3. 策略怎样通过世界模型得到一段想象经历

下面固定两个由教学策略提出的两步动作序列。A 连续向目标方向移动，B 先反向再折返。教学世界模型根据动作预测下一位置，奖励模型再按照与目标的距离给每一步评分；策略没有直接看到预先写好的回报。

为了看清数据流，这里使用一维位置。真实 Dreamer 4 使用视觉表示、动作条件 Transformer 和学习得到的奖励头。


In [ ]:
imagined_start = 0.20  # 设置两条想象轨迹共同的真实记录起点
imagined_goal = 0.80  # 设置当前任务希望到达的位置
imagined_actions = np.array([[0.18, 0.18], [-0.10, 0.12]])  # 固定教学策略提出的两组两步动作

def teaching_world_model(position, action):  # 定义可直接核对的一维教学世界模型
    return float(np.clip(position + 0.9 * action, 0.0, 1.0))  # 预测动作造成的下一位置并限制范围

def teaching_reward_model(position):  # 定义根据预测位置计算任务奖励的教学模型
    return float(1.0 - abs(imagined_goal - position) / 0.6)  # 越接近目标就给出越高奖励

def teaching_value_model(position):  # 定义未继续展开部分的教学价值估计
    return float(0.4 * max(0.0, 1.0 - abs(imagined_goal - position) / 0.6))  # 用较小权重估计更远未来

imagined_positions = []  # 保存两条轨迹中世界模型生成的位置
imagined_rewards = []  # 保存奖励模型对每个生成位置的评分
for action_sequence in imagined_actions:  # 分别展开动作序列 A 和 B
    position = imagined_start  # 每条想象都从同一个真实记录起点开始
    positions = [position]  # 保存当前轨迹的起点
    rewards = []  # 保存当前轨迹逐步产生的奖励
    for action in action_sequence:  # 沿策略给出的两步动作向前展开
        position = teaching_world_model(position, action)  # 由世界模型生成动作之后的位置
        reward = teaching_reward_model(position)  # 由奖励模型评价刚生成的结果
        positions.append(position)  # 保存生成位置供后续动作继续使用
        rewards.append(reward)  # 保存本步奖励供回报计算
    imagined_positions.append(positions)  # 保存完整想象位置序列
    imagined_rewards.append(rewards)  # 保存完整想象奖励序列
imagined_positions = np.asarray(imagined_positions)  # 整理两条位置轨迹
imagined_rewards = np.asarray(imagined_rewards)  # 整理两条奖励轨迹
tail_values = np.array([teaching_value_model(path[-1]) for path in imagined_positions])  # 估计两步之后尚未展开的未来价值
gamma = 0.9  # 设置越远奖励权重越低的折扣系数
returns = imagined_rewards[:, 0] + gamma * imagined_rewards[:, 1] + gamma**2 * tail_values  # 汇总两步预测奖励和尾部价值
baseline = float(returns.mean())  # 用两条回报平均值作为当前教学价值基准
advantages = returns - baseline  # 判断每条想象高于还是低于当前预期
for name, positions, rewards, result, advantage in zip(['A', 'B'], imagined_positions, imagined_rewards, returns, advantages):  # 逐条显示想象数据流
    print(f'轨迹 {name}：生成位置 {positions.round(3)}，预测奖励 {rewards.round(3)}，回报 {result:.3f}，优势 {advantage:+.3f}')  # 输出动作之后真正用于学习的数值
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))  # 并排显示生成位置和回报
for name, positions in zip(['A', 'B'], imagined_positions):  # 绘制两条想象位置轨迹
    axes[0].plot(range(3), positions, marker='o', label=f'trajectory {name}')  # 显示世界模型逐步生成的位置
axes[0].axhline(imagined_goal, color='black', linestyle=':', label='goal')  # 标出任务目标
axes[0].set(xlabel='imagined environment step', ylabel='predicted position', title='World-model rollout')  # 说明左图是环境时间上的想象
axes[0].legend(fontsize=8)  # 区分两条轨迹与目标
axes[1].bar(['A', 'B'], returns, color=['#497f99', '#b67d4c'])  # 显示两条想象轨迹的累计回报
axes[1].axhline(baseline, color='black', linestyle='--', label='value baseline')  # 标出用于判断优势正负的基准
axes[1].set(ylabel='illustrative return', title='Return and advantage')  # 说明右图是教学回报
axes[1].legend(fontsize=8)  # 标明价值基准
fig.tight_layout()  # 避免标题与标签重叠
plt.show()  # 输出想象轨迹与回报对照


### 从动作到优势，中间经过了哪些步骤

左图中的位置不是机器人真实执行结果，而是教学世界模型根据动作依次生成的结果。每一步生成位置再进入奖励模型，最后与尾部价值一起形成回报。右图的黑线是当前价值基准：A 高于黑线，因此优势为正；B 低于黑线，因此优势为负。

这里固定两组动作是为了逐项比较。真正的想象训练会让策略按当前概率抽取动作，并从许多真实记录起点展开。若世界模型或奖励模型预测错误，回报和优势也会随之错误。


## 4. 优势怎样改变策略概率

最后在同一个起点设置 A、B 两种第一步动作。策略开始时各给一半概率；A 的优势为正，B 的优势为负。更新目标鼓励 A、抑制 B，同时用冻结的初始策略作为行为参考，限制概率一次偏离过远。

这一步更新的是策略参数，不会重新训练教学世界模型，也不是第 16 章在当前时刻运行 CEM。


In [ ]:
logits = torch.zeros(2, requires_grad=True)  # 用两个可学习数控制当前状态的动作概率
prior = torch.tensor([0.5, 0.5])  # 固定进入想象训练前的行为先验
optimizer = torch.optim.SGD([logits], lr=0.2)  # 设置教学用概率更新步长
history = []  # 保存更新过程中两种动作的概率
for update in range(25):  # 多次使用同一教学反馈观察更新方向
    log_probs = torch.log_softmax(logits, dim=0)  # 将可学习分数变成对数概率
    probs = log_probs.exp()  # 得到真正归一化的动作概率
    kl = (probs * (log_probs - prior.log())).sum()  # 衡量当前策略偏离冻结先验的程度
    loss = -0.5 * log_probs[0] + 0.5 * log_probs[1] + 0.3 * kl  # 提高正优势动作概率并抑制负优势动作
    history.append(probs.detach().numpy().copy())  # 保存更新前的真实概率
    optimizer.zero_grad()  # 清除上一次的梯度
    loss.backward()  # 计算损失对动作分数的梯度
    optimizer.step()  # 更新动作分数而不更新任何世界模型
history = np.asarray(history)  # 整理记录供绘图
fig, ax = plt.subplots(figsize=(7, 3))  # 创建简化策略更新曲线
ax.plot(history[:, 0], label="A: positive advantage")  # 显示正优势动作的概率
ax.plot(history[:, 1], label="B: negative advantage")  # 显示负优势动作的概率
ax.set(xlabel="Illustrative update", ylabel="Probability", ylim=(0, 1))  # 明确纵轴才是策略概率
ax.legend()  # 标明正负反馈对应关系
plt.show()  # 显示策略更新方向
assert np.allclose(history.sum(axis=1), 1)  # 检查每一步仍是合法概率分布
assert history[-1, 0] > history[0, 0]  # 检查正优势反馈确实提高了动作概率


### 最后一幅图说明什么，不能说明什么

A 的概率从 0.5 逐渐提高，B 的概率相应降低，说明想象轨迹的正负反馈已经转化为策略参数变化。每个时刻两条概率之和仍为 1；代码中的 KL 项则限制策略无限远离最初各 0.5 的行为参考。

这幅图不能证明真实机器人会成功。策略学习信号完全依赖上一单元产生的想象位置和预测奖励；如果模型把危险动作错误地评为高分，策略也可能朝错误方向更新。真实系统仍要分别验证动作条件预测、奖励排序和最终闭环成功。
